In [ ]:
import json
import pandas as pd
from news_analyzer_continue import NewsAnalyzerContinue
import logging

# 配置日志
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.StreamHandler()
    ]
)

In [ ]:
# 创建analyzer实例
analyzer = NewsAnalyzerContinue()

# 加载JSON文件
news_list = analyzer.load_json_file('AIDF_FT5005.AIDF_FT5005_oil_wo_content.json')

# 打印最后两行日期，用于验证断点
if analyzer.last_processed_date and analyzer.next_processed_date:
    print(f"断点日期: {analyzer.last_processed_date} 和 {analyzer.next_processed_date}")
else:
    print("没有找到断点日期")

In [ ]:
# 找到要处理的下一条新闻
next_news = None
for news in news_list:
    date = news['date']
    if analyzer.last_processed_date and analyzer.next_processed_date:
        if date > analyzer.next_processed_date:
            next_news = news
            break
    elif analyzer.last_processed_date:
        if date > analyzer.last_processed_date:
            next_news = news
            break
    else:
        next_news = news
        break

if next_news:
    print(f"找到下一条新闻: {next_news['date']}")
    print(f"标题: {next_news['title']}")
    print(f"公司: {next_news['company_names']}")
    print(f"Tickers: {next_news['company_tickers']}")
else:
    print("没有找到下一条新闻")

In [ ]:
# 处理单条新闻
if next_news:
    # 过滤公司
    valid_names, valid_ids, valid_tickers = analyzer.filter_companies_by_ticker(
        next_news['company_names'],
        next_news['company_ids'],
        next_news['company_tickers']
    )
    
    print(f"过滤后的公司: {valid_names}")
    print(f"过滤后的Tickers: {valid_tickers}")
    
    if valid_names:
        # 调用API
        response_text = analyzer.call_claude_api(
            next_news['title'],
            next_news['content'],
            valid_names,
            valid_ids,
            valid_tickers
        )
        
        # 解析响应
        market_impact, mentions = analyzer.parse_api_response(response_text)
        
        print("\nAPI响应:")
        print(f"市场影响: {market_impact}")
        print(f"提及情况: {mentions}")
    else:
        print("没有有效的公司，跳过处理")

In [ ]:
# 验证company_info.csv中的ticker
print("\n验证company_info.csv中的ticker:")
for ticker in next_news['company_tickers']:
    ticker_base = ticker.split(" ")[0]
    exists = ticker_base in analyzer.company_info['tic'].values
    print(f"{ticker} -> {ticker_base}: {'存在' if exists else '不存在'}")